========================================================= <br>
Spark Certification Study Notebook
Bronze Layer - Data Ingestion and Initial Processing <br>
=========================================================

# Import Environment

In [0]:
# Import environment variables
from setup.env import *

from pyspark.sql import functions as F
from pyspark.sql.types import *

# Bronze Layer: Raw Data Ingestion

<br>Goal</br>

The Bronze layer is responsible for ingesting raw data from the landing zone into managed Delta tables with minimal transformation.

This layer ensures:
- raw data traceability
- schema enforcement
- reproducibility
- reliable ingestion

Bronze tables act as the single source of truth for downstream layers.

Input Data

The data generator produces the following datasets:
```
| Dataset     | Format  |
| ----------- | ------- |
| customers   | CSV     |
| products    | JSON    |
| orders      | Parquet |
| order_items | CSV     |

```
<b>Output Tables</b>

The Bronze layer will create the following tables:
- spark_cert_catalog.bronze.customers
- spark_cert_catalog.bronze.products
- spark_cert_catalog.bronze.orders
- spark_cert_catalog.bronze.order_items

## TASK 1: Create bronze_customers Ingestion DataFrame

<b> Why </b>

The first step of any data pipeline is reliable ingestion of raw data.

Using an explicit schema with StructType prevents:
- schema drift
- type inference errors
- inconsistent data types
- Explicit schemas also improve Spark performance.

<b> End Goal </b>

Create a DataFrame that reads the customers CSV dataset using an explicit schema.

This DataFrame will later be used to create the Bronze customers table.

<b> Key Concepts: </b>
- StructType
- StructField
- spark.read
- CSV ingestion
- schema enforcement

Your Task

1 - Print:
```
"Starting Bronze Layer: Customers Ingestion"
```

2 -Define the following schema (all nullable=True):
```
customer_id → StringType
email → StringType
gender → StringType
country → StringType
age → IntegerType
created_at → TimestampType
```

3 - Read the dataset using:
```
CUSTOMERS_PATH
```

4 - Store the result in:
```
customers_raw_df
```

### Your Solution

In [0]:
# Write here your solution

### Suggested Solution

In [0]:
print("=" * 50)
print("Starting Bronze Layer: Customers Ingestion")

In [0]:
customers_schema = F.StructType([
    StructField("customer_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("country", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("created_at", TimestampType(), True)
])

customers_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(customers_schema)
    .load(CUSTOMERS_PATH)
)

## TASK 2: Validate the DataFrame

<b> Why <b>

After ingestion, it is important to inspect the DataFrame structure.

Understanding the schema and data distribution helps detect:
- type issues
- null values
- unexpected fields

<b> End Goal <b>

Inspect the DataFrame structure and preview the data.

<b> Key Concepts <b>
- printSchema()
- display()
- Spark DataFrame inspection

<b> Your Task <b>

1 - Print the schema of the DataFrame <br>
2 - Display the first 10 rows

### Your Solution

### Suggested Solution

In [0]:
customers_raw_df.printSchema()

In [0]:

customers_raw_df.limit(10).display()

## TASK 3: Ingest Orders Dataset (Parquet)

<b> Why <b>

Parquet is a columnar storage format optimized for analytics.

Unlike CSV, Parquet stores:
- schema metadata
- column statistics
- compressed data

This allows Spark to read Parquet more efficiently.

<b> End Goal <b>

Create a DataFrame reading the orders dataset stored in Parquet format.

<b> Key Concepts <b>
- Parquet format
- Schema inference
- Spark DataFrame ingestion

<b> Your Task <b>

1 - Read the dataset using:
```
ORDERS_PATH
```

2 - Store the result in:
```
orders_raw_df
```

3 - Display the dataset.

### Your Solution

### Suggested Solution

In [0]:
orders_raw_df = spark.read.parquet(ORDERS_PATH)
orders_raw_df.limit(5).display()

## TASK 4: Ingest order_items CSV Dataset

<b> Why <b>

CSV is one of the most common ingestion formats in data pipelines.

However CSV has no schema enforcement, therefore defining a schema is a best practice.

<b> End Goal <b>

Create a DataFrame reading the order_items dataset with an explicit schema.

Key Concepts
- StructType
- StructField
- schema enforcement
- CSV ingestion

<b> Your Task <b>

Define the schema:
```
order_item_id → StringType
order_id → StringType
product_id → StringType
quantity → IntegerType
price → DoubleType
```

Then read the dataset from:
```
ORDER_ITEMS_PATH
```

Store the DataFrame in:
```
order_items_raw_df
```

### Your Solution

### Suggested Solution

In [0]:
order_items_schema = StructType([
    StructField("order_item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True)
])

order_items_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(order_items_schema)
    .load(ORDER_ITEMS_PATH)
)

# Another way to read the data:
# order_items_raw_df = spark.read.csv(ORDER_ITEMS_PATH, header=True, schema=order_items_schema)

# Another way to show dataframes
order_items_raw_df.show(10, truncate=False)

## TASK 5: Ingest Nested JSON Dataset (products)

<b> Why <b>

JSON files often contain nested structures such as:
- arrays
- structs
- arrays of structs

Spark can process nested data efficiently when using explicit schemas.

<b> End Goal <b>

Create a DataFrame reading the products JSON dataset with a nested schema.

<b> Key Concepts <b>
- StructType
- ArrayType
- nested JSON
- schema enforcement
- Your Task

Define the following schema:
```
product_id → StringType
name → StringType
category → StringType
price → DoubleType

attributes → struct
    color → StringType
    size → StringType

tags → array<string>

variants → array<struct>
    variant_id → StringType
    stock → IntegerType
```

Read the dataset from:
```
PRODUCTS_PATH
```
Store the result in:
```
products_raw_df
```

### Your Solution

### Suggested Solution

In [0]:
products_schema = StructType([

    StructField("product_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),

    StructField(
        "attributes",
        StructType([
            StructField("color", StringType(), True),
            StructField("size", StringType(), True)
        ])
    ),

    StructField(
        "tags",
        ArrayType(StringType())
    ),

    StructField(
        "variants",
        ArrayType(
            StructType([
                StructField("variant_id", StringType(), True),
                StructField("stock", IntegerType(), True)
            ])
        )
    )
])

products_raw_df = (
    spark.read
    .schema(products_schema)
    .json(PRODUCTS_PATH)
)

products_raw_df.limit(10).display()

## TASK 6: Add Metadata Columns

<b> Why <b>

Bronze tables often include metadata columns to track data lineage.

Examples:
- ingestion_timestamp
- source_file

This improves traceability and debugging.

<b> End Goal <b>

Add a column:
```
ingestion_timestamp
```
Using:
```
current_timestamp()
```

<b> Key Concepts <b>
```
withColumn
current_timestamp
data lineage
```

<b> Your Task <b>

Add ingestion_timestamp to:
- customers_raw_df
- orders_raw_df
- products_raw_df
- order_items_raw_df

### Your Solution

### Suggested Solution

In [0]:
customers_bronze_df = customers_raw_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

orders_bronze_df = orders_raw_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

products_bronze_df = products_raw_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

order_items_bronze_df = order_items_raw_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

In [0]:
customers_bronze_df.limit(2).display()

## TASK 7: Write Bronze Delta Tables

<b> Why <b>

After ingesting raw data into DataFrames, we must persist the data into the Lakehouse.

Spark allows writing data using multiple write modes, which define how existing data should be handled.

Understanding write behavior is critical for building reliable pipelines and is frequently tested in the Databricks Spark certification.

Bronze tables are typically append-only, but Spark supports multiple write strategies depending on the use case.

<b> End Goal <b>
Write all Bronze DataFrames as Delta tables inside the Bronze schema.

Tables:
```
bronze.customers
bronze.orders
bronze.products
bronze.order_items
```
Use:
```
CATALOG_NAME
SCHEMA_BRONZE
```

<b> Key Concepts <b>
- DataFrameWriter
- write modes
- partitionBy
- Delta Lake
- schema evolution
- save vs saveAsTable


### Write Modes in Spark

Spark supports four write modes.

```
| Mode                       | Behavior                        | When to Use                   |
| -------------------------- | ------------------------------- | ----------------------------- |
| `overwrite`                | Replaces existing data          | Rebuilding a table            |
| `append`                   | Adds new records                | Incremental ingestion         |
| `ignore`                   | Skips write if table exists     | Idempotent pipelines          |
| `error` or `errorIfExists` | Throws an error if table exists | Prevent accidental overwrites |

```

Example:
```
df.write.mode("append")
```

### Writing Methods in Spark

Spark provides two primary ways to persist DataFrames.

```
| Method          | Description                                | Use Case                |
| --------------- | ------------------------------------------ | ----------------------- |
| `save()`        | Writes data to a filesystem path           | Data lakes, raw storage |
| `saveAsTable()` | Registers a managed table in the metastore | Lakehouse tables        |

```

Example using save():
```
df.write.format("delta").save("/path/to/data")
```

Example using saveAsTable():
```
df.write.format("delta").saveAsTable("catalog.schema.table")
```

### Partitioning Data

Partitioning improves query performance by reducing the amount of data scanned.

Common partition columns:
- country
- date
- event_type

Example:
```
.partitionBy("country")
```

This creates directory structures like:
```
country=USA/
country=Brazil/
country=Germany/
```

Partitioning is very common in Bronze and Silver layers.

### Schema Evolution Options

Spark and Delta Lake support schema evolution.

Two useful options are:
```
Spark and Delta Lake support schema evolution.

Two useful options are:
```

Example:
```
.option("mergeSchema", "true")
```
Example:
```
.option("overwriteSchema", "true")
```
These options are useful when:
- new fields appear
- schema changes between batches

### Your Task

1 - Write the customers Bronze table <br> 
<b> Requirements: <b>
- Format: delta
- Mode: overwrite
- Partition by: country
- Enable schema merge
- Use saveAsTable()

2 - Write the orders Bronze dataset <br>
<b> Requirements: <b>
- Mode: append
- Use save() instead of saveAsTable()
- Path: BRONZE_PATH + "/orders"

3 - Write the products Bronze table <br>
<b> Requirements: <b>
- Format: delta
- Mode: overwrite
- Partition by: category
- Use saveAsTable()

4 - Write the order_items Bronze table <br>
<b> Requirements: <b>
- Format: delta
- Mode: overwrite
- Enable overwriteSchema
- Use saveAsTable()

### Your Solution

Use as many cells as you want to solve this.

### Suggested Solution

In [0]:
# Customers Bronze Table
customers_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("country") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_BRONZE}.customers")

In [0]:
# Orders Bronze Dataset (Filesystem)
orders_bronze_df.write \
    .format("delta") \
    .mode("append") \
    .save(f"{BRONZE_PATH}/orders")

In [0]:
# Products Bronze Table
products_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("category") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_BRONZE}.products")

In [0]:
# Order Items Bronze Table
order_items_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_BRONZE}.order_items")

## Task 8: Persist Additional Bronze Tables

<b> Goal <b>

Persist additional datasets into the Bronze layer to simulate a more realistic Lakehouse ingestion stage.

In real-world pipelines, the Bronze layer typically stores all raw ingested datasets, even if they are not immediately used.

<b> Your task <b>

1 - Read the Delta Files

2 - Persist Dataset orders_df into Bronze Tables

### Your Solution

### Suggested Solution

In [0]:
orders_bronze_df = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
orders_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_BRONZE}.orders")

## TASK 9: Query Bronze Tables Using Spark SQL

<b> Why <b>

Spark allows querying Delta tables using Spark SQL.

This is common in analytics workflows.

<b> Your Task <b>

Query the customers table.

### Your Solution

### Suggested Solution

In [0]:
%sql
SELECT *
FROM spark_cert_catalog.bronze.customers
LIMIT 10

## TASK 10: Inspect Execution Plan

<b> Why <b>

Spark builds a <b> DAG (Directed Acyclic Graph) </b>  for query execution.

Understanding execution plans helps debug performance issues.

<b> Your Task </b>

Inspect the execution plan of:
- customers_bronze_df

### Your Solution

### Suggested Solution

In [0]:
customers_bronze_df.explain(True)

In [0]:
print("Ending Bronze Layer: Customers Ingestion")
print("=" * 50)